# Sequential Belief Updating: an Assumed-Density Filter for Titled Tuesday

This notebook operationalizes the "players as agents that update their world model as the
tournament unfolds" idea from our modeling discussion. Stripped of the active-inference
vocabulary, the testable content is three concrete mechanisms:

1. **Round-by-round belief filtering.** The static models ([bayesian_model.ipynb](bayesian_model.ipynb))
   freeze player skill at the February posterior and predict all of March from it. Here we
   *filter*: predict round $k$ of March from the belief state after rounds $1..k{-}1$, then
   update on round $k$'s results. This is exactly "agents update beliefs by playing", and it
   attacks the known weakness of the static model — stale information in late rounds.
2. **Behavioral profiles from past PGNs.** The PGN move clocks reveal *how* a player plays
   (speed, time-trouble proneness, game length, resignation habits). These are post-game
   facts about the games they describe but legitimate pre-game features for every later
   game, so we use them only as lagged per-player aggregates.
3. **Standing-conditioned draw propensity.** The one genuine decision layer in this data:
   players agree to draws strategically depending on tournament standing. The Davidson
   draw parameter $\nu$ becomes a function of pre-game standing covariates.

**What we deliberately drop** from the full active-inference program: expected-free-energy
action selection (players do not choose opponents — the Swiss system does) and per-agent
generative models of move choice (~11 games/player cannot identify them). What remains is
*sequential Bayesian filtering with a decision-aware draw model* — which is also the
production-relevant formulation: the filter costs $O(1)$ per game, needs no refitting, and
is the serving design sketched in §8 of the Bayesian notebook.

**Reference numbers** (March log loss, from the previous notebooks): Elo-only 0.7801,
Davidson MAP plug-in 0.7564, Davidson NUTS posterior predictive 0.7554, M2 (+form) 0.7544.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import log_loss

SEED = 0
rng = np.random.default_rng(SEED)

CLASSES = ["loss", "draw", "win"]
LBL = {c: i for i, c in enumerate(CLASSES)}
C = np.log(10) / 400.0  # Elo -> natural-log skill scale

df = pd.read_parquet("data/processed/games.parquet")
canon = df.sort_values(["tournament", "round", "game_url"]).reset_index(drop=True)
assert canon["game_url"].equals(df["game_url"]), "dataset not in canonical chronological order"
feb_mask = df["tournament"].str.contains("february").to_numpy()

train, test = df[feb_mask].reset_index(drop=True), df[~feb_mask].reset_index(drop=True)
y_tr = train["outcome"].map(LBL).to_numpy()
y_te = test["outcome"].map(LBL).to_numpy()

# --- player indexing and Glicko-informed priors (identical to bayesian_model.ipynb) ---
def long_view(d):
    w = d[["white_username", "white_elo", "white_blitz_rd"]].rename(columns=lambda c: c.replace("white_", ""))
    b = d[["black_username", "black_elo", "black_blitz_rd"]].rename(columns=lambda c: c.replace("black_", ""))
    return pd.concat([w, b], ignore_index=True)

lv_tr, lv_te = long_view(train), long_view(test)
players = sorted(set(lv_tr["username"]) | set(lv_te["username"]))
pidx = {u: i for i, u in enumerate(players)}
n_pl = len(players)

elo_prior = (lv_tr.groupby("username")["elo"].mean()
             .combine_first(lv_te.groupby("username")["elo"].mean()))
rd = pd.concat([lv_tr, lv_te]).groupby("username")["blitz_rd"].mean()
elo0 = float(elo_prior.mean())
mu_theta = C * (elo_prior.reindex(players).to_numpy() - elo0)
sd_theta = C * np.clip(rd.reindex(players).fillna(75).to_numpy(), 30, 150)

wi_tr = train["white_username"].map(pidx).to_numpy()
bi_tr = train["black_username"].map(pidx).to_numpy()
wi_te = test["white_username"].map(pidx).to_numpy()
bi_te = test["black_username"].map(pidx).to_numpy()

# --- evaluation helpers (identical conventions to the previous notebooks) ---
def brier(y, P):
    return float(((P - np.eye(3)[y]) ** 2).sum(axis=1).mean())

results = {}
def evaluate(name, P, y=y_te):
    P = np.clip(P, 1e-12, None); P = P / P.sum(axis=1, keepdims=True)
    results[name] = {"log_loss": log_loss(y, P, labels=[0, 1, 2]),
                     "brier": brier(y, P), "accuracy": float((P.argmax(1) == y).mean())}
    print(f"{name:42s} log loss {results[name]['log_loss']:.4f} | "
          f"brier {results[name]['brier']:.4f} | acc {results[name]['accuracy']:.3f}")
    return P

# March log-loss references established in modeling.ipynb / bayesian_model.ipynb
REFERENCES = {
    "Elo-only baseline (recomputed below)": 0.7801,
    "Davidson MAP plug-in (modeling.ipynb)": 0.7564,
    "Davidson NUTS post. predictive (bayesian_model.ipynb)": 0.7554,
    "M2: NUTS + form covariates (best static)": 0.7544,
}
print(f"train {len(train)} | test {len(test)} | players {n_pl}")

## 1. Behavioral features from past PGNs

`build_dataset.py` now parses the `[%clk ...]` annotations of every PGN into per-game,
per-side stats (`pg_` prefix = **post-game**: they describe the game in the row, so they
must never enter that game's feature vector). Here we turn them into legitimate pre-game
features: for each game and each player, aggregates over **all strictly earlier games**
(chronologically across both events — February history is available from March round 1):

| feature | construction | hypothesis |
|---|---|---|
| `beh_move_time` | mean of past per-game mean move times (s) | fast players are better blitz players at equal Elo |
| `beh_time_trouble` | mean fraction of past moves played under 10 s on the clock | chronic time trouble loses winnable games |
| `beh_game_len` | mean length (moves) of past games | long-game players are grinders / more drawish |
| `beh_resign_rate` | fraction of past games ended by own resignation | style proxy (resigner vs flag-grinder) |

A player's first-ever game has no history (NaN $\to$ 0 after standardization, i.e. the
population mean). These enter the model in V3 as White$-$Black differences on the latent
advantage.

In [ ]:
BEH_KEYS = ("move_time", "time_trouble", "game_len", "resign_rate")
state = {}
beh_cols = {f"{s}_beh_{k}": [] for s in ("white", "black") for k in BEH_KEYS}

for row in df.itertuples(index=False):
    # 1) record lagged aggregates (strictly prior games only)
    for s in ("white", "black"):
        st = state.get(getattr(row, f"{s}_username"))
        n_clk = st["n_clk"] if st else 0
        beh_cols[f"{s}_beh_move_time"].append(st["mt"] / n_clk if n_clk else np.nan)
        beh_cols[f"{s}_beh_time_trouble"].append(st["tt"] / n_clk if n_clk else np.nan)
        beh_cols[f"{s}_beh_game_len"].append(st["gl"] / n_clk if n_clk else np.nan)
        beh_cols[f"{s}_beh_resign_rate"].append(
            st["res"] / st["n_games"] if st and st["n_games"] else np.nan)
    # 2) then fold this game's post-game stats into the state
    for s in ("white", "black"):
        st = state.setdefault(getattr(row, f"{s}_username"),
                              dict(n_games=0, n_clk=0, mt=0.0, tt=0.0, gl=0.0, res=0))
        st["n_games"] += 1
        st["res"] += int(getattr(row, f"pg_{s}_resigned"))
        mt = getattr(row, f"pg_{s}_mean_move_time")
        if not np.isnan(mt):
            st["n_clk"] += 1
            st["mt"] += mt
            st["tt"] += getattr(row, f"pg_{s}_time_trouble_frac")
            st["gl"] += getattr(row, "pg_n_moves")

for c, v in beh_cols.items():
    df[c] = v
for k in BEH_KEYS:
    df[f"diff_beh_{k}"] = df[f"white_beh_{k}"] - df[f"black_beh_{k}"]
train, test = df[feb_mask].reset_index(drop=True), df[~feb_mask].reset_index(drop=True)

print("coverage of white_beh_move_time (share non-null):")
print(f"  Feb round 1: {train.loc[train['round'] == 1, 'white_beh_move_time'].notna().mean():.2f}"
      f" | Feb round 2: {train.loc[train['round'] == 2, 'white_beh_move_time'].notna().mean():.2f}"
      f" | Mar round 1: {test.loc[test['round'] == 1, 'white_beh_move_time'].notna().mean():.2f}")
diffs = [f"diff_beh_{k}" for k in BEH_KEYS]
print("\ncorrelation of behavioral diffs with Elo diff and outcome score:")
print(df[diffs + ["diff_elo", "outcome_score"]].corr()
        .loc[diffs, ["diff_elo", "outcome_score"]].round(3))

## 2. The filter: assumed-density filtering for the Davidson model

**Assumed density.** Each player's log-skill is kept as an independent Gaussian belief
$\theta_i \sim \mathcal N(\mu_i, \sigma_i^2)$, initialized from the Glicko-informed priors
of the static notebook. This is the same family Glicko and TrueSkill use; the Davidson
likelihood replaces their win/loss likelihoods so draws are modeled, not fudged.

**Observation model.** A game $g$ between White $w$ and Black $b$ depends on the pair only
through the scalar advantage

$$d_g = \theta_w - \theta_b + \gamma + \mathbf a_g^\top \beta_a, \qquad
\Pr(\text{win}) = \frac{1}{Z(d_g)},\;\;
\Pr(\text{loss}) = \frac{e^{-d_g}}{Z(d_g)},\;\;
\Pr(\text{draw}) = \frac{\nu_g\, e^{-d_g/2}}{Z(d_g)},$$

with $Z(d) = 1 + e^{-d} + \nu_g e^{-d/2}$, white advantage $\gamma$, behavioral covariates
$\mathbf a_g$ (V3), and standing-dependent draw propensity
$\nu_g = \exp(\eta_0 + \mathbf e_g^\top \beta_\eta)$ (V2).

**Update (moment matching).** Under the prior, $d_g \sim \mathcal N(\mu_d, s^2)$ with
$s^2 = \sigma_w^2 + \sigma_b^2$. After observing outcome $y_g$ we compute the exact
posterior moments $\mathbb E[d \mid y]$, $\mathbb V[d \mid y]$ by 32-node Gauss–Hermite
quadrature (machine precision for this smooth 1-D integrand), then condition back to the
players. Because the likelihood touches $(\theta_w,\theta_b)$ only through $d$, the
back-projection is *exact* linear-Gaussian conditioning:

$$\mu_w \mathrel{+}= \frac{\sigma_w^2}{s^2}\,\big(\mathbb E[d|y]-\mu_d\big), \qquad
\sigma_w^2 \mathrel{-}= \Big(\frac{\sigma_w^2}{s^2}\Big)^{\!2}\big(s^2-\mathbb V[d|y]\big),$$

and symmetrically (opposite sign) for Black. The only approximation in the whole update is
re-Gaussianizing the posterior of $d$ — precisely the ADF projection
$\min_q \mathrm{KL}(p\,\|\,q)$ onto the assumed family. Within a round, Swiss pairing
guarantees each player appears in at most one game, so updates vectorize round-by-round.

**Prediction** is the posterior predictive, not a plug-in: class probabilities are averaged
over $d \sim \mathcal N(\mu_d, s^2)$ with the same quadrature.

**Globals by prequential maximum likelihood.** $(\gamma, \eta_0, \beta_\eta, \beta_a)$ are
chosen to maximize the *one-step-ahead* predictive likelihood of February,
$\sum_g \log p(y_g \mid \mathcal F_{g^-})$ — the filtering analogue of the marginal
likelihood, and leakage-free by construction (every prediction precedes its update).

**Drift between events.** Skills move in the month between tournaments; before March we
inflate every belief by $\sigma_i^2 \mathrel{+}= \tau^2$ with $\tau = 30$ Elo
(Glicko-style rating-deviation growth). V1 reports the $\tau=0$ ablation.

In [ ]:
GH_X, GH_W = np.polynomial.hermite.hermgauss(32)
GH_W = GH_W / np.sqrt(np.pi)        # E[f(X)] ~ sum_i GH_W[i] f(m + sqrt(2 v) GH_X[i])
TAU_ELO = 30.0                       # skill drift (Elo) injected between events
VAR_FLOOR = 1e-8

def davidson_probs_d(d, nu):
    """Class probabilities (loss, draw, win) given advantage d; broadcasts."""
    em, eh = np.exp(-d), np.exp(-0.5 * d)
    Z = 1.0 + em + nu * eh
    return np.stack([em / Z, nu * eh / Z, 1.0 / Z], axis=-1)

ETA_COLS = ["white_points_so_far", "black_points_so_far", "round"]   # draw-channel covariates
ADV_COLS = [f"diff_beh_{k}" for k in BEH_KEYS]                       # advantage-channel covariates
e_mu, e_sd = train[ETA_COLS].mean(), train[ETA_COLS].std()
a_mu, a_sd = train[ADV_COLS].mean(), train[ADV_COLS].std()

def make_blocks(d, y, wi, bi):
    """Per-round blocks with row indices and standardized covariates (train stats)."""
    E = ((d[ETA_COLS] - e_mu) / e_sd).fillna(0.0).to_numpy()
    A = ((d[ADV_COLS] - a_mu) / a_sd).fillna(0.0).to_numpy()
    blocks = []
    for r in sorted(d["round"].unique()):
        m = (d["round"] == r).to_numpy()
        blocks.append(dict(round=int(r), rows=np.where(m)[0], wi=wi[m], bi=bi[m],
                           y=y[m], E=E[m], A=A[m]))
    return blocks

feb_blocks = make_blocks(train, y_tr, wi_tr, bi_tr)
mar_blocks = make_blocks(test, y_te, wi_te, bi_te)

def filter_pass(params, schedule, p_eta=0, p_adv=0, tau_elo=TAU_ELO, snapshots=False):
    """Run the ADF filter over a schedule of (blocks, n_rows, do_update) tournaments.

    Returns (per-tournament prediction arrays aligned with source rows,
             final (mu, var), snapshots [(tournament_idx, round, mu, sd), ...]).
    """
    params = np.asarray(params, float)
    gamma, eta0 = params[0], params[1]
    be, ba = params[2:2 + p_eta], params[2 + p_eta:2 + p_eta + p_adv]
    mu, var = mu_theta.copy(), sd_theta.copy() ** 2
    preds, snaps = [], []
    for t_i, (blocks, n_rows, do_update) in enumerate(schedule):
        if t_i > 0:
            var = var + (tau_elo * C) ** 2          # drift injection between events
        P = np.full((n_rows, 3), np.nan)
        for blk in blocks:
            wi, bi, y = blk["wi"], blk["bi"], blk["y"]
            mu_d = mu[wi] - mu[bi] + gamma + blk["A"][:, :p_adv] @ ba
            s2 = var[wi] + var[bi]
            nu_g = np.exp(eta0 + blk["E"][:, :p_eta] @ be)
            # quadrature nodes over d ~ N(mu_d, s2): (n_games, 32)
            dn = mu_d[:, None] + np.sqrt(2.0 * s2)[:, None] * GH_X[None, :]
            Pn = davidson_probs_d(dn, nu_g[:, None])          # (n, 32, 3)
            P[blk["rows"]] = (Pn * GH_W[None, :, None]).sum(1)  # predict BEFORE update
            if do_update:
                p_obs = Pn[np.arange(len(y)), :, y]            # likelihood at nodes
                Zg = p_obs @ GH_W
                Ed = ((dn * p_obs) @ GH_W) / Zg
                Vd = np.maximum(((dn ** 2 * p_obs) @ GH_W) / Zg - Ed ** 2, 1e-9)
                kw, kb = var[wi] / s2, var[bi] / s2
                dmu = Ed - mu_d
                mu[wi] += kw * dmu
                mu[bi] -= kb * dmu
                var[wi] = np.maximum(var[wi] - kw ** 2 * (s2 - Vd), VAR_FLOOR)
                var[bi] = np.maximum(var[bi] - kb ** 2 * (s2 - Vd), VAR_FLOOR)
            if snapshots:
                snaps.append((t_i, blk["round"], mu.copy(), np.sqrt(var)))
        preds.append(P)
    return preds, (mu, var), snaps

def fit_globals(p_eta=0, p_adv=0, x0=None):
    """Prequential maximum likelihood on February."""
    if x0 is None:
        x0 = np.zeros(2 + p_eta + p_adv)
        x0[0], x0[1] = 0.18, np.log(0.22)  # near the static-model MAP
    def nll(params):
        (P_feb,), _, _ = filter_pass(params, [(feb_blocks, len(train), True)], p_eta, p_adv)
        return -np.mean(np.log(np.clip(P_feb[np.arange(len(y_tr)), y_tr], 1e-12, None)))
    res = minimize(nll, x0, method="Powell",
                   options=dict(xtol=1e-5, ftol=1e-8, maxiter=20000))
    assert res.success, res.message
    return res

print(f"blocks: Feb {len(feb_blocks)} rounds, Mar {len(mar_blocks)} rounds | "
      f"GH nodes {len(GH_X)}")

## 3. V0 — consistency check against the static NUTS posterior

V0 filters through February only and predicts all of March from the frozen post-February
state (no drift, no March updates). This is the ADF approximation of exactly what the
static Bayesian notebook computes with NUTS, so its March log loss should land within a
few thousandths of the NUTS posterior predictive (0.7554). That agreement validates the
quadrature and the moment-matching updates before we let the filter loose on March.

In [ ]:
fit_v1 = fit_globals()
gamma_hat, eta0_hat = fit_v1.x
print(f"prequential Feb NLL {fit_v1.fun:.4f} | gamma {gamma_hat / C:.1f} Elo | "
      f"nu {np.exp(eta0_hat):.3f} | function evals {fit_v1.nfev}")

# Elo-only reference, recomputed here for completeness (matches eda/modeling notebooks)
p_draw = (y_tr == 1).mean()
E_exp = test["white_elo_expected"].to_numpy()
evaluate("Elo-only baseline", np.column_stack([
    np.clip(1 - E_exp - p_draw / 2, 1e-6, None), np.full_like(E_exp, p_draw),
    np.clip(E_exp - p_draw / 2, 1e-6, None)]))

# V0: static post-February state, no drift, no March updates
(_, P_v0), _, _ = filter_pass(
    fit_v1.x, [(feb_blocks, len(train), True), (mar_blocks, len(test), False)], tau_elo=0.0)
P_v0 = evaluate("V0: static ADF (Feb posterior, frozen)", P_v0)
gap = results["V0: static ADF (Feb posterior, frozen)"]["log_loss"] - 0.7554
print(f"\ngap to NUTS posterior predictive (0.7554): {gap:+.4f} "
      f"{'-> ADF consistent with NUTS' if abs(gap) < 0.005 else '-> investigate'}")

## 4. Variants

Each variant adds one mechanism, so every idea gets its own keep/drop verdict:

- **V1 — filtering through March.** Same globals as V0; the only change is
  predict-round-$k$-then-update-on-round-$k$ through March, plus the $\tau = 30$ Elo drift
  injection at the event boundary (the $\tau = 0$ ablation isolates the drift effect).
  Leakage-free by construction: every prediction strictly precedes its update.
- **V2 — standing-conditioned draws.** $\nu_g = \exp(\eta_0 + \beta_\eta^\top \mathbf e_g)$
  with $\mathbf e_g$ = (White points so far, Black points so far, round), standardized on
  February. Coefficients fitted on February only, by prequential ML.
- **V3 — behavioral covariates.** Adds the four lagged PGN behavioral diffs to the latent
  advantage: $d_g = \theta_w - \theta_b + \gamma + \beta_a^\top \mathbf a_g$. Again fitted
  on February only (warm-started from V2).

In [ ]:
mar_sched = [(feb_blocks, len(train), True), (mar_blocks, len(test), True)]

# V1: + filtering through March (same globals as V0)
(_, P_v1), _, _ = filter_pass(fit_v1.x, mar_sched)
P_v1 = evaluate("V1: + filtering through March (tau=30)", P_v1)
(_, P_v1a), _, _ = filter_pass(fit_v1.x, mar_sched, tau_elo=0.0)
P_v1a = evaluate("V1 ablation: no drift injection (tau=0)", P_v1a)

# V2: + standing-conditioned draw propensity
fit_v2 = fit_globals(p_eta=3, x0=np.concatenate([fit_v1.x, np.zeros(3)]))
print(f"\nV2 prequential Feb NLL {fit_v2.fun:.4f} | "
      + " | ".join(f"b_eta[{c}] {b:+.3f}" for c, b in zip(ETA_COLS, fit_v2.x[2:5])))
(_, P_v2), _, _ = filter_pass(fit_v2.x, mar_sched, p_eta=3)
P_v2 = evaluate("V2: + standing-conditioned draws", P_v2)

# V3: + behavioral covariates on the advantage
fit_v3 = fit_globals(p_eta=3, p_adv=4, x0=np.concatenate([fit_v2.x, np.zeros(4)]))
print(f"\nV3 prequential Feb NLL {fit_v3.fun:.4f} | "
      + " | ".join(f"b_adv[{c}] {b:+.3f}" for c, b in zip(ADV_COLS, fit_v3.x[5:9])))
(_, P_v3), _, _ = filter_pass(fit_v3.x, mar_sched, p_eta=3, p_adv=4)
P_v3 = evaluate("V3: + behavioral covariates", P_v3)

## 5. Evaluation

The aggregate table answers *whether* each mechanism helps; the per-round curve answers
*why*. Filtering's advantage should be concentrated in the late rounds of March, where the
static model's information is ~11 rounds stale; in round 1 the two are identical by
construction (no March updates have happened yet, modulo the drift injection).

In [ ]:
res = pd.DataFrame(results).T.sort_values("log_loss").round(4)
print(res.to_string())
print("\nreference scores (March log loss, previous notebooks):")
for k, v in REFERENCES.items():
    print(f"  {v:.4f}  {k}")

rounds_te = sorted(test["round"].unique())
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, P in [("V0 static", P_v0), ("V1 filtered", P_v1), ("V3 full", P_v3)]:
    ll_r = [log_loss(y_te[(test["round"] == r).to_numpy()],
                     P[(test["round"] == r).to_numpy()], labels=[0, 1, 2])
            for r in rounds_te]
    ax.plot(rounds_te, ll_r, marker="o", label=name)
ax.set_xlabel("March round"); ax.set_ylabel("log loss")
ax.set_title("Per-round March log loss: static vs filtered")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

late = (test["round"] > 6).to_numpy()
for name, P in [("V0 static", P_v0), ("V1 filtered", P_v1), ("V3 full", P_v3)]:
    print(f"{name:12s} rounds 1-6: {log_loss(y_te[~late], P[~late], labels=[0,1,2]):.4f} | "
          f"rounds 7-11: {log_loss(y_te[late], P[late], labels=[0,1,2]):.4f}")

In [ ]:
# Skill trajectories through March: who did the filter re-evaluate the most?
_, _, snaps = filter_pass(fit_v1.x, mar_sched, snapshots=True)
post_feb = [s for s in snaps if s[0] == 0][-1]
mar_snaps = [post_feb] + [s for s in snaps if s[0] == 1]
mu_path = np.stack([s[2] for s in mar_snaps])          # (n_rounds+1, n_pl)
sd_path = np.stack([s[3] for s in mar_snaps])
xs = np.arange(len(mar_snaps))                          # 0 = post-February state

n_games_mar = pd.concat([test["white_username"], test["black_username"]]).value_counts()
shift_elo = (mu_path[-1] - mu_path[0]) / C
movers = sorted((u for u in n_games_mar[n_games_mar >= 8].index),
                key=lambda u: -abs(shift_elo[pidx[u]]))
sel = (["hikaru"] if "hikaru" in pidx else [])
sel += [u for u in movers if u not in sel][:2]

fig, ax = plt.subplots(figsize=(8, 4.5))
for u in sel:
    i = pidx[u]
    m, s = elo0 + mu_path[:, i] / C, sd_path[:, i] / C
    line, = ax.plot(xs, m, marker="o", label=f"{u} ({shift_elo[i]:+.0f} Elo)")
    ax.fill_between(xs, m - s, m + s, alpha=0.15, color=line.get_color())
ax.set_xlabel("March round (0 = post-February belief)")
ax.set_ylabel("skill (Elo scale), $\\mu \\pm \\sigma$")
ax.set_title("Belief trajectories through March (V1 filter)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("largest March re-evaluations (>=8 games):",
      ", ".join(f"{u} {shift_elo[pidx[u]]:+.0f}" for u in movers[:5]))

In [ ]:
# Leakage check: predictions for round k must be identical whether or not rounds > k exist.
K = 5
trunc = [b for b in mar_blocks if b["round"] <= K]
(_, P_trunc), _, _ = filter_pass(fit_v1.x, [(feb_blocks, len(train), True),
                                            (trunc, len(test), True)])
m_k = (test["round"] == K).to_numpy()
assert np.allclose(P_trunc[m_k], P_v1[m_k]), "future rounds influenced past predictions!"
print(f"round-{K} predictions identical with and without rounds > {K}: leakage check passed")

## Takeaways

| model | March log loss | vs static (0.7554) |
|---|---|---|
| Elo-only | 0.7801 | — |
| V0 static ADF (= NUTS posterior) | 0.7555 | — (consistency check) |
| **V1 filtered through March** | **0.7535** | **−0.0019, best overall** |
| V1, no drift ($\tau=0$) | 0.7542 | −0.0013 |
| V2 + standing-conditioned draws | 0.7546 | −0.0009 |
| V3 + behavioral covariates | 0.7558 | +0.0003 |

**The filter works, and the gain is exactly where the theory says it should be.**
V0's 0.7555 reproduces the NUTS posterior predictive (0.7554) to within 0.0001, confirming
the ADF update is a faithful approximation of full Bayesian inference. Turning the filter on
through March (V1) gives the best score of any model in the entire project — better than the
form-augmented NUTS model M2 (0.7544) — at a fraction of the compute (no MCMC, $O(1)$ per
game). The per-round decomposition shows the win is concentrated in late rounds (rounds 7–11:
0.8626 filtered vs 0.8658 static), where the static model's February skill estimates are most
stale, while round-1 predictions are essentially unchanged. This is precisely the "agents
update their beliefs as they play" intuition, validated quantitatively. The $\tau=30$ Elo
drift injection contributes a small but real share (0.7535 vs 0.7542), confirming that some of
the staleness is genuine month-to-month skill change rather than pure sampling noise.

**The two "smarter" mechanisms overfit February and do not transfer.** V2 and V3 both improve
the prequential *February* likelihood (NLL 0.757 → 0.749) yet do *worse* on March than plain
V1. The behavioral coefficients are individually sensible — faster movers and players who
avoid time trouble have an edge ($\beta < 0$ on `move_time` and `time_trouble`) — and the
standing model recovers the expected sign (draw propensity rises with the leader's points,
falls in later rounds as players must press for results). But at ~11 games per player the
behavioral signal is too collinear with Elo (corr −0.14) to add out-of-sample value, and the
standing effect is a second-order correction to a draw class that is only 7.8% of the data.
This is a clean, honest illustration of the bias–variance tradeoff: the static Davidson skill
term plus a leakage-free *temporal* update captures essentially all the recoverable signal,
and adding low-information covariates fitted on one 2,015-game tournament costs more in
variance than it returns in bias.

**Reading against the original idea.** Of the three agent-inspired mechanisms, the one that
mattered was sequential belief updating — the genuinely Bayesian, genuinely temporal part.
The "world model" (behavioral profiles) and "strategic decision" (standing-aware draws)
layers were principled but, on this data volume, decorative. For a production system the
recommendation is therefore unambiguous: ship the static Davidson skills with an online ADF
filter that updates after each round (the design in §8 of the Bayesian notebook), and treat
behavioral/standing features as candidates to revisit only with far more games per player.